# Phase 3 · Bank Complaint Router

Use a T4 runtime under the Algo Shank account. In the Files pane, upload `complaint-router-20261106-035015-773607.zip`, `val.csv`, and `eval.py` directly to `/content`. Add `LANGSMITH_API_KEY` in Colab Secrets and enable notebook access to publish both experiments. The validation split is only used for evaluation.

Verify the GPU with `!nvidia-smi -L`. Upload the three files before running setup, and download `/content/phase3-eval-results.zip` before ending the session. If the key was unavailable during scoring, the final cell publishes saved results from the same runtime. See Week5 COLAB.md for the exact bundle size and the separate April procedure.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from shutil import copyfileobj
from time import monotonic, sleep
from zipfile import ZipFile

from google.colab import userdata

root = Path("/content")
bundle = root / "complaint-router-20261106-035015-773607.zip"
val = root / "val.csv"
script = root / "eval.py"
for path in (val, script):
    if not path.is_file():
        raise FileNotFoundError(f"Upload {path.name} via the Files pane.")

expected_bytes = 3457143110
chunk_size = 256 * 1024**2
parts = [root / f"router-part-{index:02d}" for index in range(13)]
part_sizes = [chunk_size] * 12 + [expected_bytes - 12 * chunk_size]
deadline = monotonic() + 5400
while not (bundle.is_file() and bundle.stat().st_size == expected_bytes):
    if all(part.is_file() and part.stat().st_size == size for part, size in zip(parts, part_sizes)):
        bundle = root / "reassembled-router.zip"
        with bundle.open("wb") as destination:
            for part in parts:
                with part.open("rb") as source:
                    copyfileobj(source, destination)
        break
    if monotonic() > deadline:
        raise TimeoutError("Model upload did not finish in 90 minutes.")
    sleep(10)

artifacts = root / "phase2_artifacts"
with ZipFile(bundle) as archive:
    archive.extractall(artifacts)

try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("LangSmith secret is unavailable. Metrics will still be saved.")

subprocess.run([sys.executable, "-m", "pip", "install",
                "transformers>=4.57,<6", "scikit-learn", "langsmith", "python-dotenv"], check=True)

In [ ]:
from shutil import make_archive

output = root / "eval_results"
command = [sys.executable, "-u", str(script), "--val", str(val),
           "--merged", str(artifacts / "merged_model"), "--out", str(output)]
try:
    subprocess.run(command, check=True)
finally:
    if output.exists():
        print("Download from Files:", make_archive(str(root / "phase3-eval-results"), "zip", output))

If metrics finished before the key was available, enable the secret and run this cell to publish the recorded predictions without running inference again.

In [ ]:
try:
    key = userdata.get("LANGSMITH_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("Enable LANGSMITH_API_KEY in Colab Secrets and rerun this cell to publish the experiments.")
else:
    subprocess.run([*command, "--log-only"], env={**os.environ, "LANGSMITH_API_KEY": key}, check=True)
    print("Download from Files:", make_archive(str(root / "phase3-eval-results"), "zip", output))